# Lab Day 10: Finetune Qwen3-VL-4B (LoRA) để định vị hư hại mặt đường

> Tinh chỉnh mô hình Vision-Language Model Qwen3-VL-4B bằng QLoRA trên tập dữ liệu ảnh dashcam.
> Môi trường thực thi khuyến nghị: **Google Colab T4 GPU** (16 GB VRAM).
> Xem chi tiết tài liệu tại `README.md`, `GUIDE.md`, `RUBRIC.md` và hoàn thiện báo cáo tại `REPORT.md`.

## 0. Môi trường (Colab T4)

In [ ]:
# Runtime → Change runtime type → T4 GPU; chạy từ thư mục gốc repository.
!nvidia-smi -L
# Lab dùng bitsandbytes; gỡ torchao có sẵn để tránh xung đột với PEFT.
%pip uninstall -y torchao
%pip -q install -r requirements-colab.txt pytest matplotlib tqdm
!python -m pytest -q
# Sau khi thay phiên bản thư viện đã import: Runtime → Restart session.

In [ ]:
# Kaggle credentials: Colab Secrets hoặc ~/.kaggle/kaggle.json.
import os
try:
    from google.colab import userdata
except ImportError:
    print('Chạy local: dùng ~/.kaggle/kaggle.json hoặc biến môi trường Kaggle.')
else:
    try:
        username = userdata.get('KAGGLE_USERNAME')
        key = userdata.get('KAGGLE_KEY')
        os.environ['KAGGLE_USERNAME'] = username
        os.environ['KAGGLE_KEY'] = key
        print('✓ Đã nạp Kaggle credentials từ Colab Secrets!')
    except (userdata.SecretNotFoundError, userdata.NotebookAccessError) as e:
        print(f'Chưa nạp được Colab Secrets: {e}. Kiểm tra quyền truy cập Secret hoặc dùng kaggle.json.')

## 1. Dữ liệu: tải và chia split

In [ ]:
# Tải tập dữ liệu từ Kaggle và chia thành 3 tập train, val, test theo phiên quay (session)
# Phân chia theo phiên quay giúp ngăn chặn triệt để rò rỉ dữ liệu giữa train và test
!python scripts/prepare_data.py --kaggle

In [ ]:
# Đọc thông tin và trực quan hoá các mẫu dữ liệu
import json
from pathlib import Path
from PIL import Image, ImageDraw
import matplotlib.pyplot as plt

# Đọc thông tin phân bố split từ data/splits/meta.json
meta_path = Path('data/splits/meta.json')
if meta_path.exists():
    meta = json.loads(meta_path.read_text(encoding='utf-8'))
    print(f"Dataset version: {meta.get('dataset_version')}")
    print('Phân bố số lượng ảnh và nhãn theo từng split:')
    for split_name, cnt in meta.get('counts', {}).items():
        print(f" - {split_name:5s}: {cnt['images']:4d} ảnh | {cnt['boxes']}")

def read_jsonl(path):
    with open(path, encoding='utf-8') as f:
        return [json.loads(line) for line in f if line.strip()]

train_records = read_jsonl('data/splits/train.jsonl')
print(f"\nTổng số mẫu ảnh trong tập train: {len(train_records)}")

# Trực quan hoá 4 ảnh mẫu kèm bounding box (chuyển đổi từ YOLO [0, 1] sang toạ độ pixel)
fig, axes = plt.subplots(2, 2, figsize=(14, 8))
axes = axes.flatten()

color_map = {
    'pothole': 'red',     # ổ gà
    'crack': 'yellow',    # vết nứt
    'manhole': 'cyan'     # nắp cống
}

sample_records = [r for r in train_records if len(r['objects']) > 0][:4]
for idx, rec in enumerate(sample_records):
    img_path = Path('data') / rec['image_path']
    img = Image.open(img_path).convert('RGB')
    draw = ImageDraw.Draw(img)
    w_img, h_img = img.size
    
    for obj in rec['objects']:
        label = obj['label']
        xc, yc, bw, bh = obj['yolo']
        # YOLO (tâm xc, yc, kích thước bw, bh) -> Toạ độ pixel góc trên-trái và dưới-phải
        x1 = (xc - bw / 2) * w_img
        y1 = (yc - bh / 2) * h_img
        x2 = (xc + bw / 2) * w_img
        y2 = (yc + bh / 2) * h_img
        
        c = color_map.get(label, 'green')
        draw.rectangle([x1, y1, x2, y2], outline=c, width=3)
        draw.text((x1 + 2, max(0, y1 - 12)), label, fill=c)
        
    axes[idx].imshow(img)
    axes[idx].set_title(f"{rec['image_id']} ({len(rec['objects'])} objects)")
    axes[idx].axis('off')

plt.tight_layout()
plt.show()

## 2. Nhãn: YOLO → JSON thang 0–1000

In [ ]:
# Hàm chuyển đổi nhãn YOLO sang định dạng JSON grounding thang 0-1000

def yolo_to_box(label: str, yolo: list):
    """
    Chuyển nhãn YOLO [xc, yc, w, h] chuẩn hoá [0, 1] sang toạ độ nguyên [x1, y1, x2, y2]
    trên thang chuẩn 0-1000 (quy ước grounding của Qwen3-VL).
    """
    xc, yc, w, h = yolo
    x1 = (xc - w / 2) * 1000.0
    y1 = (yc - h / 2) * 1000.0
    x2 = (xc + w / 2) * 1000.0
    y2 = (yc + h / 2) * 1000.0
    
    # Làm tròn thành số nguyên và kẹp (clamp) trong khoảng [0, 1000]
    ix1 = max(0, min(1000, int(round(x1))))
    iy1 = max(0, min(1000, int(round(y1))))
    ix2 = max(0, min(1000, int(round(x2))))
    iy2 = max(0, min(1000, int(round(y2))))
    
    return {'label': label, 'bbox_2d': [ix1, iy1, ix2, iy2]}

def objects_to_answer(objects: list) -> str:
    """
    Chuyển danh sách objects thành chuỗi JSON:
    - Sắp xếp các box theo thứ tự (y1, x1): từ trên xuống dưới, từ trái sang phải
      để tạo thứ tự ổn định giúp mô hình học nhất quán.
    - Trả về JSON string không dấu cách thừa (separators=(',', ':')).
    """
    boxes = [yolo_to_box(o['label'], o['yolo']) for o in objects]
    boxes.sort(key=lambda b: (b['bbox_2d'][1], b['bbox_2d'][0]))
    return json.dumps(boxes, separators=(',', ':'))

# Định nghĩa câu lệnh Prompt chuẩn hướng dẫn mô hình phát hiện 3 lớp hư hại mặt đường
PROMPT = (
    'Detect road damages in this image. '
    'Identify potholes, cracks, and manholes. '
    'Return a JSON list of objects, each with \'label\' and \'bbox_2d\': [x1, y1, x2, y2] '
    'using normalized coordinates on a 0-1000 scale.'
)

# Kiểm tra thử hàm chuyển đổi với mẫu đầu tiên
sample = train_records[0]
sample_answer = objects_to_answer(sample['objects'])
print('Prompt:\n', PROMPT)
print('\nJSON Answer tương ứng của mẫu đầu tiên:\n', sample_answer)

## 3. E0: zero-shot

In [ ]:
# Nạp model gốc Qwen3-VL-4B lượng tử hoá 4-bit (NF4) và suy luận Zero-Shot trên tập test
import torch
from transformers import AutoProcessor, BitsAndBytesConfig
from transformers import Qwen3VLForConditionalGeneration as VLMModelClass
from tqdm.auto import tqdm

def make_quantization_config():
    # Transformers 5.19 đối chiếu từ đầu đường dẫn: phải có model.visual.
    return BitsAndBytesConfig(
        load_in_4bit=True, bnb_4bit_quant_type='nf4',
        bnb_4bit_compute_dtype=torch.float16, bnb_4bit_use_double_quant=True,
        llm_int8_skip_modules=['model.visual', 'visual', 'lm_head'])

def assert_vision_not_quantized(model):
    # Chạy TRƯỚC get_peft_model / requires_grad=True, kể cả E1 và inference.
    vision = [(n, p) for n, p in model.named_parameters()
              if n.startswith('visual.') or '.visual.' in n]
    if not vision:
        raise ValueError('Không tìm thấy visual trong cây model.')
    bad = [f'{n}: {type(p).__name__}, {p.dtype}' for n, p in vision
           if hasattr(p, 'quant_state') or not p.is_floating_point()]
    if bad:
        raise TypeError('ViT/projector đã bị lượng tử hoá 4-bit. Không thể mở gradient. '
                        "Nạp lại base với skip_modules=['model.visual', 'visual', 'lm_head']; "
                        'không dùng .float() để giải lượng tử.\n' + '\n'.join(bad[:10]))

def load_base_model():
    # Tạo config mới mỗi lần để không dùng quantization_config cũ trong Colab.
    model = VLMModelClass.from_pretrained(
        model_id, quantization_config=make_quantization_config(),
        dtype=torch.float16, device_map={'': 0})
    assert_vision_not_quantized(model)
    return model

quantization_config = make_quantization_config()

assert torch.cuda.is_available(), 'Hãy chọn GPU runtime trong Colab.'
model_id = 'Qwen/Qwen3-VL-4B-Instruct'
print(f'Đang tải processor và model: {model_id} ...')

processor = AutoProcessor.from_pretrained(model_id)
# Đặt padding_side = 'left' cho Decoder khi suy luận batch
processor.tokenizer.padding_side = 'left'



In [ ]:
model_e0 = load_base_model()
model_e0.eval()
print('✓ Đã tải xong model E0 lên GPU!')

In [ ]:
# Chạy suy luận Zero-Shot trên tập test (311 ảnh)
test_records = read_jsonl('data/splits/test.jsonl')
output_e0_path = Path('work/e0.jsonl')
output_e0_path.parent.mkdir(parents=True, exist_ok=True)

batch_size = 4
e0_predictions = []

print(f'Bắt đầu suy luận E0 trên {len(test_records)} ảnh test (batch size = {batch_size})...')

for i in tqdm(range(0, len(test_records), batch_size)):
    batch = test_records[i : i + batch_size]
    batch_images = [Image.open(Path('data') / r['image_path']).convert('RGB') for r in batch]
    
    batch_messages = [
        [
            {
                'role': 'user',
                'content': [
                    {'type': 'image', 'image': img},
                    {'type': 'text', 'text': PROMPT}
                ]
            }
        ]
        for img in batch_images
    ]
    
    inputs = processor.apply_chat_template(
        batch_messages,
        tokenize=True,
        add_generation_prompt=True,
        return_dict=True,
        return_tensors='pt',
        padding=True
    ).to('cuda')
    
    prompt_len = inputs.input_ids.shape[1]
    
    with torch.no_grad():
        out = model_e0.generate(
            **inputs,
            max_new_tokens=256,
            do_sample=False
        )
        
    # Cắt bỏ phần prompt, chỉ lấy các token mới sinh ra
    gen_tokens = out[:, prompt_len:]
    decoded = processor.batch_decode(gen_tokens, skip_special_tokens=True)
    
    for r, raw_text in zip(batch, decoded):
        e0_predictions.append({'image_id': r['image_id'], 'raw': raw_text.strip()})

# Ghi kết quả vào work/e0.jsonl
with open(output_e0_path, 'w', encoding='utf-8') as f:
    for row in e0_predictions:
        f.write(json.dumps(row, ensure_ascii=False) + '\n')

print(f'✓ Đã ghi {len(e0_predictions)} kết quả E0 vào {output_e0_path}')

# Xem thử 3 kết quả thô đầu tiên
print('\n--- Ví dụ 3 output thô từ E0 ---')
for p in e0_predictions[:3]:
    print(f"[{p['image_id']}]: {p['raw']}")

In [ ]:
# Chấm điểm E0 bằng trọng tài score.py
!python scripts/score.py --pred work/e0.jsonl --name E0 --out report/evidence/e0.json

In [ ]:
# Có thể chạy lại ô này mà không gặp NameError.
import gc
if 'model_e0' in globals():
    del model_e0
# Xoá cả tensor output cuối cùng của E0.
for tensor_name in ('inputs', 'out', 'gen_tokens'):
    globals().pop(tensor_name, None)
gc.collect()
torch.cuda.empty_cache()
print('✓ Đã giải phóng VRAM của E0!')

## 4. Dựng dữ liệu huấn luyện: chat template và mask loss

In [ ]:
# Hàm encode_example và collate_fn cho huấn luyện SFT với Mask Loss
from torch.utils.data import Dataset, DataLoader

class RoadDamageDataset(Dataset):
    def __init__(self, records, data_root='data'):
        self.records = records
        self.data_root = Path(data_root)

    def __len__(self):
        return len(self.records)

    def __getitem__(self, idx):
        rec = self.records[idx]
        image_path = self.data_root / rec['image_path']
        image = Image.open(image_path).convert('RGB')
        answer = objects_to_answer(rec['objects'])
        return image, PROMPT, answer

def encode_example(image, prompt, answer, processor):
    """
    Mã hoá một mẫu huấn luyện:
    - prompt_enc: chỉ phần prompt của user kèm generation prompt.
    - full_enc: toàn bộ chuỗi (user prompt + assistant answer).
    - labels: gán nhãn -100 cho toàn bộ token thuộc prompt và ảnh;
      chỉ tính loss trên các token của câu trả lời.
    """
    user_msg = {
        'role': 'user',
        'content': [
            {'type': 'image', 'image': image},
            {'type': 'text', 'text': prompt}
        ]
    }
    assistant_msg = {
        'role': 'assistant',
        'content': [{'type': 'text', 'text': answer}]
    }

    prompt_enc = processor.apply_chat_template(
        [user_msg],
        add_generation_prompt=True,
        tokenize=True,
        return_dict=True,
        return_tensors='pt'
    )
    
    full_enc = processor.apply_chat_template(
        [user_msg, assistant_msg],
        add_generation_prompt=False,
        tokenize=True,
        return_dict=True,
        return_tensors='pt'
    )

    n = prompt_enc.input_ids.shape[1]
    # Kiểm tra prompt là tiền tố chuẩn xác của chuỗi đầy đủ
    assert torch.equal(full_enc.input_ids[0, :n], prompt_enc.input_ids[0]), 'Prompt mismatch!'

    # Tạo labels để mask loss: chỉ tính loss trên câu trả lời của assistant
    labels = full_enc.input_ids.clone()
    labels[:, :n] = -100

    item = {
        'input_ids': full_enc.input_ids[0],
        'attention_mask': full_enc.attention_mask[0],
        'labels': labels[0]
    }
    if 'pixel_values' in full_enc:
        item['pixel_values'] = full_enc.pixel_values
    if 'image_grid_thw' in full_enc:
        item['image_grid_thw'] = full_enc.image_grid_thw
    if 'mm_token_type_ids' in full_enc:
        item['mm_token_type_ids'] = full_enc.mm_token_type_ids[0]

    return item

def collate_fn(batch_items, pad_token_id):
    """
    Hàm ghép lô (collate):
    - Đệm (pad) input_ids, attention_mask, labels đến độ dài lớn nhất trong batch.
    - Nối (cat) pixel_values và image_grid_thw theo chiều 0 (KHÔNG đệm).
    """
    max_len = max(len(x['input_ids']) for x in batch_items)
    
    input_ids_list = []
    attention_mask_list = []
    labels_list = []
    mm_token_types = []
    
    for x in batch_items:
        pad_len = max_len - len(x['input_ids'])
        input_ids_list.append(torch.cat([x['input_ids'], torch.full((pad_len,), pad_token_id, dtype=torch.long)]))
        attention_mask_list.append(torch.cat([x['attention_mask'], torch.zeros(pad_len, dtype=torch.long)]))
        labels_list.append(torch.cat([x['labels'], torch.full((pad_len,), -100, dtype=torch.long)]))
        if 'mm_token_type_ids' in x:
            mm_token_types.append(torch.cat([x['mm_token_type_ids'], torch.zeros(pad_len, dtype=torch.long)]))

    batch = {
        'input_ids': torch.stack(input_ids_list),
        'attention_mask': torch.stack(attention_mask_list),
        'labels': torch.stack(labels_list),
    }
    if mm_token_types:
        batch['mm_token_type_ids'] = torch.stack(mm_token_types)
        
    if 'pixel_values' in batch_items[0]:
        batch['pixel_values'] = torch.cat([x['pixel_values'] for x in batch_items], dim=0)
    if 'image_grid_thw' in batch_items[0]:
        batch['image_grid_thw'] = torch.cat([x['image_grid_thw'] for x in batch_items], dim=0)
        
    return batch

# Kiểm tra thử encode_example với mẫu đầu tiên
sample_img = Image.open(Path('data') / train_records[0]['image_path']).convert('RGB')
sample_enc = encode_example(sample_img, PROMPT, objects_to_answer(train_records[0]['objects']), processor)

# In số token ảnh và giải mã phần câu trả lời (token labels != -100)
active_labels = sample_enc['labels'][sample_enc['labels'] != -100]
print(f"Tổng chiều dài token chuỗi: {len(sample_enc['input_ids'])}")
print(f"Số token thực sự tính loss: {len(active_labels)}")
print('Nội dung giải mã phần tính loss:')
print(processor.tokenizer.decode(active_labels))

## 5. Setup LoRA / đóng băng cho E1, E2, E3

In [ ]:
# LoRA + projector: mọi trọng số trainable phải ở FP32 khi dùng GradScaler.
from peft import LoraConfig, get_peft_model
import re

def is_projector(name):
    return 'visual.merger.' in name or 'visual.deepstack_merger_list.' in name

def ensure_trainable_fp32(model):
    for name, param in model.named_parameters():
        if param.requires_grad:
            if not param.is_floating_point() or hasattr(param, 'quant_state'):
                raise TypeError(f'Tham số trainable bị lượng tử hoá: {name}. Kiểm tra skip_modules.')
            # Chạy TRƯỚC khi tạo optimizer; không đổi dtype giữa các bước train.
            param.data = param.data.float()
            param.grad = None
    assert_trainable_fp32(model)

def assert_trainable_fp32(model, check_gradients=False):
    bad = []
    for name, param in model.named_parameters():
        if param.requires_grad and param.dtype != torch.float32:
            bad.append(f'{name}: weight={param.dtype}')
        if check_gradients and param.grad is not None and param.grad.dtype != torch.float32:
            bad.append(f'{name}: grad={param.grad.dtype}')
    if bad:
        raise TypeError('GradScaler cần tham số/gradient trainable FP32:\n' + '\n'.join(bad[:10]))

def save_projector(model, path):
    state = {name: p.detach().cpu().clone() for name, p in model.named_parameters()
             if is_projector(name)}
    if not state:
        raise ValueError('Không tìm thấy projector để lưu.')
    torch.save(state, path)

def load_projector(model, path):
    # Thiếu/sai checkpoint phải dừng; không đánh giá bằng projector gốc.
    state = torch.load(path, map_location='cpu', weights_only=True)
    params = {name: p for name, p in model.named_parameters() if is_projector(name)}
    if set(state) != set(params):
        raise ValueError(f'Projector checkpoint không khớp: thiếu={set(params)-set(state)}, '
                         f'thừa={set(state)-set(params)}')
    for name, p in params.items():
        if state[name].shape != p.shape:
            raise ValueError(f'Sai shape projector: {name}')
    # Giữ FP32 như lúc train; generate phải chạy trong autocast.
    with torch.no_grad():
        for name, p in params.items():
            p.data = p.data.float()
            p.copy_(state[name].to(device=p.device, dtype=p.dtype))

def apply_setup(model, setup: str, r: int = 16, alpha: int = 32, dropout: float = 0.05):
    setup = setup.upper()
    if setup not in ('E1', 'E2', 'E3'):
        raise ValueError(f'Setup không hợp lệ: {setup}')
    assert_vision_not_quantized(model)
    print(f'\n--- Thiết lập mô hình cho {setup} ---')
    for param in model.parameters():
        param.requires_grad = False
    model.enable_input_require_grads()

    llm = r'language_model.*\.(q_proj|k_proj|v_proj|o_proj|gate_proj|up_proj|down_proj)'
    vit = r'visual\.blocks.*\.(qkv|proj|linear_fc1|linear_fc2)'
    target_regex = rf'.*({llm}|{vit})$' if setup == 'E3' else rf'.*{llm}$'
    matched = [n for n, _ in model.named_modules() if re.fullmatch(target_regex, n)]
    expected = model.config.text_config.num_hidden_layers * 7
    if setup == 'E3':
        expected += model.config.vision_config.depth * 4
    if len(matched) != expected:
        raise ValueError(f'LoRA targets: tìm được {len(matched)}, cần {expected}. Kiểm tra cây model.')
    print(f'LoRA targets: {len(matched)} modules')
    lora_config = LoraConfig(r=r, lora_alpha=alpha, lora_dropout=dropout,
                            target_modules=target_regex, bias='none', task_type='CAUSAL_LM')
    peft_model = get_peft_model(model, lora_config, autocast_adapter_dtype=True)
    if setup in ('E2', 'E3'):
        projector_count = 0
        for name, param in peft_model.named_parameters():
            if is_projector(name):
                if hasattr(param, 'quant_state') or not param.is_floating_point():
                    raise TypeError(f'Projector bị lượng tử hoá: {name}. Phải nạp lại base.')
                param.requires_grad = True
                projector_count += param.numel()
        if not projector_count:
            raise ValueError('Không tìm thấy projector. Kiểm tra cây model.')
        print(f'✓ Đã mở Projector ({projector_count:,} tham số)')
    ensure_trainable_fp32(peft_model)
    # PEFT hiệu chỉnh số tham số 4-bit; numel() đơn thuần đếm storage đóng gói.
    trainable, total = peft_model.get_nb_trainable_parameters()
    print(f'Trainable FP32: {trainable:,} / {total:,} ({100*trainable/total:.2f}%)')
    return peft_model

**Sau khi cập nhật notebook trên Colab:** chạy lại ô imports/processor và các hàm
`make_quantization_config`, `load_base_model` ở mục 3, rồi các ô định nghĩa
`apply_setup`, `train_experiment`, `evaluate_experiment` trước khi chạy riêng E2.
Ô nạp model E0 đã được tách riêng, có thể bỏ qua nếu không chạy lại E0.
Lỗi `only Tensors of floating point and complex dtype can require gradients`
nghĩa là ViT/projector có thể đã bị lượng tử hoá: cần nạp lại base với
`llm_int8_skip_modules=['model.visual', 'visual', 'lm_head']`.
Nếu lỗi cũ giữ tensor trong traceback và gây OOM, restart session rồi nạp lại
dữ liệu, processor, imports và các hàm. Nếu thư mục E2 có checkpoint chạy dở,
dùng `run_name='e2_retry'` khi train, kèm `evidence_filename='e2.json'` khi evaluate.
E1/E0 chạy bằng config cũ có thể đã lượng tử hoá cả ViT; chạy lại bằng config mới
để so sánh E0–E4 nhất quán, dùng run_name mới để giữ checkpoint cũ.


## 6. Vòng huấn luyện

In [ ]:
# Chỉ đếm bước optimizer thực sự cập nhật; bỏ qua bước tràn gradient.
import gc
import time
import math
from torch.amp import autocast, GradScaler
from transformers import get_cosine_schedule_with_warmup, set_seed

def optimizer_update(model, optimizer, scaler, scheduler, params):
    assert_trainable_fp32(model, check_gradients=True)
    scaler.unscale_(optimizer)
    torch.nn.utils.clip_grad_norm_(params, 1.0)
    old_scale = scaler.get_scale()
    scaler.step(optimizer)
    scaler.update()
    optimizer.zero_grad(set_to_none=True)
    updated = scaler.get_scale() >= old_scale
    if updated:
        scheduler.step()
    return updated

def train_experiment(setup: str, lora_r: int = 16, lora_alpha: int = 32,
                     max_steps: int = 300, lr: float = 2e-4, run_name: str = None,
                     batch_size: int = 2, grad_accum_steps: int = 4, seed: int = 42):
    setup = setup.upper()
    if setup not in ('E1', 'E2', 'E3'):
        raise ValueError(f'Setup không hợp lệ: {setup}')
    if min(max_steps, batch_size, grad_accum_steps) <= 0 or not train_records:
        raise ValueError('Số bước/batch/grad_accum phải > 0 và train không rỗng.')
    run_name = run_name or setup.lower()
    if not re.fullmatch(r'[a-zA-Z0-9_-]+', run_name):
        raise ValueError('run_name chỉ chứa chữ, số, dấu _ hoặc -.')
    save_dir = Path('work') / run_name
    # Không ghi đè thí nghiệm khác hoặc checkpoint đã chạy thành công.
    if save_dir.exists() and any(save_dir.iterdir()):
        raise FileExistsError(f'{save_dir} đã có checkpoint. Dùng run_name mới để chạy lại.')
    assert torch.cuda.is_available(), 'Cần GPU runtime.'
    set_seed(seed)
    gc.collect()
    torch.cuda.empty_cache()
    torch.cuda.reset_peak_memory_stats()
    start_time = time.time()
    base_model = model = optimizer = scheduler = scaler = None
    batch = outputs = loss = None
    params = []
    pbar = None
    try:
        base_model = load_base_model()
        model = apply_setup(base_model, setup, r=lora_r, alpha=lora_alpha)
        model.train()
        model.gradient_checkpointing_enable(gradient_checkpointing_kwargs={'use_reentrant': False})
        model.config.use_cache = False
        assert_trainable_fp32(model)
        params = [p for p in model.parameters() if p.requires_grad]
        trainable_count = sum(p.numel() for p in params)
        optimizer = torch.optim.AdamW(params, lr=lr, weight_decay=0.0)
        scheduler = get_cosine_schedule_with_warmup(
            optimizer, num_warmup_steps=int(0.05*max_steps), num_training_steps=max_steps)
        # Scale ban đầu thấp hơn mặc định để giảm tràn gradient ở model tiền huấn luyện.
        scaler = GradScaler('cuda', init_scale=1024.0)
        pad_id = processor.tokenizer.pad_token_id
        if pad_id is None:
            raise ValueError('Tokenizer thiếu pad_token_id.')
        def custom_collate(rows):
            encoded = [encode_example(img, prompt, ans, processor) for img, prompt, ans in rows]
            return collate_fn(encoded, pad_id)
        loader = DataLoader(RoadDamageDataset(train_records), batch_size=batch_size,
                            shuffle=True, collate_fn=custom_collate)
        data_iter = iter(loader)
        step = skipped_updates = consecutive_skips = 0
        optimizer.zero_grad(set_to_none=True)
        pbar = tqdm(total=max_steps, desc=f'Training {run_name}')
        while step < max_steps:
            running_loss = 0.0
            for _ in range(grad_accum_steps):
                try:
                    batch = next(data_iter)
                except StopIteration:
                    data_iter = iter(loader)
                    batch = next(data_iter)
                batch = {k: v.to('cuda') for k, v in batch.items()}
                with autocast('cuda', dtype=torch.float16):
                    outputs = model(**batch)
                    loss = outputs.loss / grad_accum_steps
                value = loss.detach().item()
                if not math.isfinite(value):
                    raise ValueError(f'Loss không hữu hạn tại bước {step}: {value}. Kiểm tra mask và LR.')
                scaler.scale(loss).backward()
                running_loss += value
                batch = outputs = loss = None
            # Xác nhận đường gradient qua projector/LoRA ngay lần backward đầu tiên.
            if step == 0:
                missing = [n for n, p in model.named_parameters() if p.requires_grad and p.grad is None]
                if missing:
                    raise RuntimeError('Tham số trainable không có gradient: ' + ', '.join(missing[:10]))
            if not optimizer_update(model, optimizer, scaler, scheduler, params):
                skipped_updates += 1
                consecutive_skips += 1
                if consecutive_skips >= 20:
                    raise RuntimeError('20 lần tràn gradient liên tiếp. Kiểm tra dữ liệu/FP16 trước khi chạy tiếp.')
                continue
            consecutive_skips = 0
            step += 1
            pbar.update(1)
            pbar.set_postfix(loss=f'{running_loss:.4f}', lr=f'{scheduler.get_last_lr()[0]:.2e}',
                             skipped=skipped_updates)
            if step % 100 == 0 or step == max_steps:
                save_dir.mkdir(parents=True, exist_ok=True)
                model.save_pretrained(save_dir / 'adapter')
                if setup in ('E2', 'E3'):
                    save_projector(model, save_dir / 'projector.pt')
                torch.cuda.synchronize()
                run_meta = {
                    'setup': setup, 'run_name': run_name, 'model_id': model_id,
                    'seed': seed, 'lora_r': lora_r, 'lora_alpha': lora_alpha, 'lr': lr,
                    'batch_size': batch_size, 'grad_accum_steps': grad_accum_steps,
                    'max_steps': max_steps, 'completed_steps': step,
                    'skipped_updates': skipped_updates, 'trainable_params': trainable_count,
                    'peak_vram_gb': round(torch.cuda.max_memory_allocated() / 1024**3, 3),
                    'train_minutes': round((time.time()-start_time) / 60, 2),
                    'dataset_version': json.loads(Path('data/splits/meta.json').read_text())['dataset_version'],
                }
                (save_dir / 'run.json').write_text(json.dumps(run_meta, indent=2), encoding='utf-8')
        print(f'✓ Hoàn thành {run_name}: {step} bước, checkpoint: {save_dir}')
        return run_meta
    finally:
        if pbar is not None:
            pbar.close()
        batch = outputs = loss = None
        params.clear()
        model = base_model = optimizer = scheduler = scaler = None
        gc.collect()
        torch.cuda.empty_cache()

## 7. Suy luận và chấm E1–E3

In [ ]:
# Nạp checkpoint đầy đủ, suy luận trong autocast và chấm với số liệu thật.
import gc
from peft import PeftModel
import subprocess
import sys

def validate_run_meta(meta, setup, dataset_version):
    if meta.get('setup', setup) != setup:
        raise ValueError('run.json không đúng setup.')
    if 'dataset_version' in meta and meta['dataset_version'] != dataset_version:
        raise ValueError('Checkpoint và dữ liệu hiện tại khác dataset_version.')
    if 'completed_steps' in meta and meta['completed_steps'] != meta.get('max_steps'):
        raise ValueError('Checkpoint chưa hoàn tất số bước huấn luyện.')
    for key in ('trainable_params', 'peak_vram_gb', 'train_minutes'):
        value = meta.get(key)
        if isinstance(value, bool) or not isinstance(value, (int, float)) or not math.isfinite(value) or value <= 0:
            raise ValueError(f'run.json thiếu hoặc sai số liệu thật: {key}')

def evaluate_experiment(setup: str, run_meta: dict = None, evidence_filename: str = None,
                        run_name: str = None, score_name: str = None, split: str = 'test',
                        batch_size: int = 4):
    setup = setup.upper()
    if setup not in ('E1', 'E2', 'E3') or split not in ('val', 'test'):
        raise ValueError('Setup/split không hợp lệ.')
    run_name = run_name or (run_meta or {}).get('run_name', setup.lower())
    if not re.fullmatch(r'[a-zA-Z0-9_-]+', run_name) or batch_size <= 0:
        raise ValueError('run_name/batch_size không hợp lệ.')
    save_dir = Path('work') / run_name
    # Luôn đọc metadata đi cùng checkpoint; giữ hỗ trợ checkpoint E1 cũ.
    checkpoint_meta = json.loads((save_dir / 'run.json').read_text(encoding='utf-8'))
    if run_meta is not None and run_meta != checkpoint_meta:
        raise ValueError('run_meta truyền vào không khớp run.json của checkpoint.')
    run_meta = checkpoint_meta
    dataset_version = json.loads(Path('data/splits/meta.json').read_text())['dataset_version']
    validate_run_meta(run_meta, setup, dataset_version)
    if not (save_dir / 'adapter' / 'adapter_config.json').is_file():
        raise FileNotFoundError(f'Thiếu adapter tại {save_dir}')
    if setup in ('E2', 'E3') and not (save_dir / 'projector.pt').is_file():
        raise FileNotFoundError(f'Thiếu projector.pt tại {save_dir}')
    score_name = score_name or setup
    pred_path = Path('work') / f'{run_name}{"_val" if split == "val" else ""}.jsonl'
    records = read_jsonl(f'data/splits/{split}.jsonl')
    if not records:
        raise ValueError('Split đánh giá rỗng.')
    processor.tokenizer.padding_side = 'left'
    base_model = model = inputs = out = gen_tokens = None
    gc.collect()
    torch.cuda.empty_cache()
    try:
        base_model = load_base_model()
        model = PeftModel.from_pretrained(base_model, save_dir / 'adapter')
        if setup in ('E2', 'E3'):
            load_projector(model, save_dir / 'projector.pt')
        model.eval()
        model.config.use_cache = True
        with pred_path.open('w', encoding='utf-8') as f:
            for i in tqdm(range(0, len(records), batch_size), desc=f'Eval {run_name} {split}'):
                batch = records[i:i+batch_size]
                images = [Image.open(Path('data') / r['image_path']).convert('RGB') for r in batch]
                messages = [[{'role': 'user', 'content': [
                    {'type': 'image', 'image': img}, {'type': 'text', 'text': PROMPT}]}] for img in images]
                inputs = processor.apply_chat_template(
                    messages, tokenize=True, add_generation_prompt=True, return_dict=True,
                    return_tensors='pt', padding=True).to('cuda')
                # Projector FP32 + ViT FP16 cần autocast cả khi generate.
                with torch.inference_mode(), autocast('cuda', dtype=torch.float16):
                    out = model.generate(**inputs, max_new_tokens=256, do_sample=False)
                gen_tokens = out[:, inputs.input_ids.shape[1]:]
                decoded = processor.batch_decode(gen_tokens, skip_special_tokens=True)
                for rec, text in zip(batch, decoded):
                    f.write(json.dumps({'image_id': rec['image_id'], 'raw': text.strip()}, ensure_ascii=False) + '\n')
                inputs = out = gen_tokens = None
    finally:
        model = base_model = inputs = out = gen_tokens = None
        gc.collect()
        torch.cuda.empty_cache()
    evidence_filename = evidence_filename or f'{run_name}.json'
    evidence_path = Path('report/evidence' if split == 'test' else 'work/val_evidence') / evidence_filename
    cmd = [sys.executable, 'scripts/score.py', '--pred', str(pred_path), '--name', score_name,
           '--split', split, '--out', str(evidence_path)]
    for key in ('trainable_params', 'peak_vram_gb', 'train_minutes'):
        cmd += ['--meta', f'{key}={run_meta[key]}']
    subprocess.run(cmd, check=True)
    return json.loads(evidence_path.read_text(encoding='utf-8'))

In [ ]:
# Chạy E1, E2, E3 ở các ô riêng phía dưới để chạy lại E2 không lặp E1.
# Trên T4 nếu OOM: dùng batch_size=1, grad_accum_steps=8 cho TẤT CẢ E1–E4.
# Mỗi train_experiment nạp base model sạch; E2/E3 không tiếp tục từ adapter E1.

In [ ]:
meta_e1 = train_experiment('E1')
evaluate_experiment('E1', meta_e1)


In [ ]:
meta_e2 = train_experiment('E2')
evaluate_experiment('E2', meta_e2)


In [ ]:
meta_e3 = train_experiment('E3')
evaluate_experiment('E3', meta_e3)


## 8. E4 (ablation tự chọn) và E5 (tuỳ chọn, GRPO)

In [ ]:
# E4: rank=4 và rank=64, alpha=2*r; chọn setup tốt nhất trên VAL.
# Checkpoint/output riêng: work/e4_rank_4 và work/e4_rank_64; không ghi đè E1–E3.
# Nếu vừa chạy lại bằng e2_retry, lấy run_name từ meta_e2 trong session.
run_names = {candidate: (globals().get(f'meta_{candidate.lower()}') or {}).get(
    'run_name', candidate.lower()) for candidate in ('E1', 'E2', 'E3')}
val_scores = {}
for candidate in ('E1', 'E2', 'E3'):
    val_scores[candidate] = evaluate_experiment(candidate, run_name=run_names[candidate], split='val')['micro']['f1']
best_setup = max(val_scores, key=val_scores.get)
print(f'Setup tốt nhất trên val: {best_setup}, F1={val_scores[best_setup]:.3f}')
baseline_meta = json.loads((Path('work') / run_names[best_setup] / 'run.json').read_text(encoding='utf-8'))
common = {key: baseline_meta.get(key, default) for key, default in {
    'max_steps': 300, 'lr': 2e-4, 'batch_size': 2, 'grad_accum_steps': 4, 'seed': 42
}.items()}
# Checkpoint cũ chỉ có số liệu tài nguyên: dùng cấu hình chuẩn E1–E3.
for rank in (4, 64):
    name = f'e4_rank_{rank}'
    meta_e4 = train_experiment(best_setup, lora_r=rank, lora_alpha=2*rank, run_name=name, **common)
    evaluate_experiment(best_setup, meta_e4, run_name=name, score_name='E4',
                        evidence_filename=f'{name}.json')

In [ ]:
# ==========================================================
# E5 (tuỳ chọn): CHỈ định nghĩa/test reward; ô này CHƯA huấn luyện GRPO
# ==========================================================
import math
from scripts.score import parse_output, iou_xyxy

def compute_iou_reward(completion: str, gt_boxes: list) -> float:
    """
    Hàm tính reward kiểm chứng được cho GRPO theo GUIDE bước 9:
    - Nếu JSON hỏng -> 0.0
    - Nếu không có gt và không có pred -> 1.0
    - Ghép đôi tham lam theo IoU giảm dần, chỉ nhận IoU >= 0.5
    - Phạt nếu sinh thừa box bịa đặt (qua mẫu số max(n_gt, n_pred, 1))
    """
    pred = parse_output(completion)
    if pred is None:
        return 0.0
    if len(gt_boxes) == 0 and len(pred) == 0:
        return 1.0
    if len(gt_boxes) == 0 or len(pred) == 0:
        return 0.2
        
    pairs = []
    for i, g in enumerate(gt_boxes):
        for j, p in enumerate(pred):
            if g['label'] == p['label']:
                iou_val = iou_xyxy(g['bbox'], p['bbox'])
                if iou_val >= 0.5:
                    pairs.append((iou_val, i, j))
                    
    pairs.sort(reverse=True, key=lambda x: x[0])
    used_g, used_p = set(), set()
    total_iou = 0.0
    for iou_val, i, j in pairs:
        if i not in used_g and j not in used_p:
            used_g.add(i)
            used_p.add(j)
            total_iou += iou_val
            
    reward_score = 0.2 + 0.8 * (total_iou / max(len(gt_boxes), len(pred), 1))
    return float(reward_score)

# Kiểm tra unit test cho hàm reward
perfect_gt = [{'label': 'pothole', 'bbox': (400, 400, 600, 600)}]
assert compute_iou_reward('[{"label":"pothole","bbox_2d":[400,400,600,600]}]', perfect_gt) == 1.0
assert compute_iou_reward('invalid json text', perfect_gt) == 0.0
assert compute_iou_reward('[]', []) == 1.0
assert compute_iou_reward('[]', perfect_gt) == 0.2
assert compute_iou_reward('[{"label":"crack","bbox_2d":[400,400,600,600]}]', perfect_gt) == 0.2
assert math.isclose(compute_iou_reward('[{"label":"pothole","bbox_2d":[400,400,600,600]},{"label":"pothole","bbox_2d":[0,0,100,100]}]', perfect_gt), 0.6)
print('Test 1 - Dự đoán hoàn hảo (IoU = 1.0) :', compute_iou_reward('[{"label":"pothole","bbox_2d":[400,400,600,600]}]', perfect_gt))
print('Test 2 - JSON lỗi cú pháp             :', compute_iou_reward('invalid json text', perfect_gt))
print('Test 3 - Cả hai đều rỗng (ảnh sạch)   :', compute_iou_reward('[]', []))
print('Test 4 - Sai nhãn (crack thay pothole):', compute_iou_reward('[{"label":"crack","bbox_2d":[400,400,600,600]}]', perfect_gt))


In [ ]:
# Kiểm tra toàn bộ bài nộp trước khi commit và nộp bài
# Chỉ đạt '0 FAIL' khi có đủ evidence, REPORT.md đã điền và lịch sử git theo rubric.
!python scripts/check_submission.py